# I-Environnement

In [39]:
# ============================================================
# STEP 1 — Environnement
# ============================================================
import os, math, time, csv, json
from pathlib import Path
from collections import defaultdict

#!pip install codecarbon
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import (efficientnet_b0, EfficientNet_B0_Weights)
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              f1_score, precision_score, recall_score)
import yaml

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
print(f"GPUs disponibles : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i} : {torch.cuda.get_device_name(i)} "
          f"— {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")

# Vérifier EfficientNet
try:
    _eff = efficientnet_b0(weights=None)
    print("EfficientNet-B0 : disponible ✓")
    del _eff
except Exception as e:
    print(f"EfficientNet-B0 : erreur — {e}")

class Struct:
    def __init__(self, **entries):
        self.__dict__.update(entries)


from PIL import Image

Device  : cuda
GPUs disponibles : 2
  GPU 0 : Tesla T4 — 15.6 GB
  GPU 1 : Tesla T4 — 15.6 GB
PyTorch : 2.11.0+cu128
EfficientNet-B0 : disponible ✓


# II-Config DDR, trois échelles

In [40]:
# ============================================================
# STEP 1 — Config MS-IPS DDR à TROIS échelles
#
# Image 896, échelles 128 / 64 / 32 (taille = pas, pavage exact) :
#     128 px ->  7 x  7 =  49 patchs
#      64 px -> 14 x 14 = 196 patchs
#      32 px -> 28 x 28 = 784 patchs
#   total                1029 candidats  (contre 245 à deux échelles)
#
# patch_out fixe la taille du tenseur commun envoyé à l'encodeur.
# DEUX configurations sont fournies, et il faut les deux pour pouvoir
# interpréter le résultat :
#
#   A. patch_out = 128  « convention actuelle »
#      Les échelles fines sont sur-échantillonnées vers 128, comme dans
#      le run à deux échelles. Même tenseur, même mémoire, donc
#      DIRECTEMENT COMPARABLE au run à deux échelles. Répond à :
#      « une troisième échelle plus fine améliore-t-elle le résultat ? »
#
#   B. patch_out = 32   « pyramide stricte »
#      Les échelles grossières sont sous-échantillonnées vers 32, donc
#      chaque échelle apporte ce que les autres n'ont pas : 128 donne un
#      champ large vu grossièrement, 32 donne le détail natif. Répond à :
#      « lever la domination change-t-elle l'usage des échelles ? »
#
#      ATTENTION : à patch_out = 32, le classifieur ne reçoit que
#      10 x 32 x 32 = 10 240 pixels contre 163 840 à patch_out = 128,
#      soit 16 fois moins. Une baisse d'exactitude ne pourra donc PAS
#      être attribuée aux trois échelles sans comparer à la config A.
#      C'est la raison d'être des deux runs.
# ============================================================
import yaml

_DDR3_BASE = """
# optimisation
n_epoch: 50
B: 16
B_seq: 8
n_epoch_warmup: 10
lr: 0.0001
wd: 0.1

# dataset
n_class: 5
data_dir: '/kaggle/input/datasets/samriddhibagchi/ddr-dataset-credits-to-authors/DDR-dataset'
img_size: 896
n_worker: 4
pin_memory: True
eager: True

# divers
eps: 0.000001
seed: 42
track_epochs: [0, 1]

# leviers VRAM
use_checkpoint: False
ips_encode_batch: 128

# encodeur
enc_type: 'resnet18'
pretrained: True
n_chan_in: 3
n_res_blocks: 4

# patchifier : du plus grossier au plus fin, taille = pas
scales: [128, 64, 32]

# IPS
shuffle: False
shuffle_style: 'batch'
n_token: 1
M: 10
I: 10

# agrégateur
use_pos: False
use_scale_emb: True
H: 8
D: 512
D_k: 64
D_v: 64
D_inner: 2048
attn_dropout: 0.1
dropout: 0.1

# tâches
tasks:
  task0:
    id: 0
    name: 'dr_grade'
    act_fn: 'softmax'
    metric:
      - accuracy
      - kappa
      - f1
"""

def _make(patch_out, run_name):
    d = yaml.safe_load(_DDR3_BASE)
    d['patch_out'] = patch_out
    d['run_name']  = run_name
    return Struct(**d)

# A — comparable au run à deux échelles
msips3_ddr_up_conf   = _make(128, 'msips3_ddr_up_v1')

# B — pyramide stricte
msips3_ddr_pyr_conf  = _make(32,  'msips3_ddr_pyr_v1')

# ── Récapitulatif ────────────────────────────────────────────
for conf in (msips3_ddr_up_conf, msips3_ddr_pyr_conf):
    n_per = [(conf.img_size // s) ** 2 for s in conf.scales]
    print(f"[{conf.run_name}]  patch_out = {conf.patch_out}")
    for s, n in zip(conf.scales, n_per):
        ratio = s / conf.patch_out
        kind = ('natif' if ratio == 1
                else (f'sous-échantillonné {ratio:.0f}:1' if ratio > 1
                      else f'sur-échantillonné x{1/ratio:.0f}'))
        print(f"    {s:3d} px : {conf.img_size//s:2d}x{conf.img_size//s:<2d} = {n:4d} patchs   -> {kind}")
    print(f"    total   : {sum(n_per)} candidats, M = {conf.M}, "
          f"soit {conf.M/sum(n_per):.1%} retenus")
    print(f"    tenseur : {conf.M} x {conf.n_chan_in} x {conf.patch_out} x {conf.patch_out} "
          f"= {conf.M*conf.n_chan_in*conf.patch_out**2} valeurs\n")


[msips3_ddr_up_v1]  patch_out = 128
    128 px :  7x7  =   49 patchs   -> natif
     64 px : 14x14 =  196 patchs   -> sur-échantillonné x2
     32 px : 28x28 =  784 patchs   -> sur-échantillonné x4
    total   : 1029 candidats, M = 10, soit 1.0% retenus
    tenseur : 10 x 3 x 128 x 128 = 491520 valeurs

[msips3_ddr_pyr_v1]  patch_out = 32
    128 px :  7x7  =   49 patchs   -> sous-échantillonné 4:1
     64 px : 14x14 =  196 patchs   -> sous-échantillonné 2:1
     32 px : 28x28 =  784 patchs   -> natif
    total   : 1029 candidats, M = 10, soit 1.0% retenus
    tenseur : 10 x 3 x 32 x 32 = 30720 valeurs



# III-Dataset DDR

In [41]:
# ============================================================
# Dataset DDR — retourne l'image ENTIÈRE ; le découpage multi-échelle
# est délégué à MSIPSNet.ips()
# ============================================================
import hashlib
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


class DDRDatasetMSIPS(Dataset):
    """
    DDR — rétinopathie diabétique, 5 classes (0→4).
    Retourne l'image ENTIÈRE (3, img_size, img_size).
    Le patchifier multi-échelle est délégué à MSIPSNet.ips().
    Classe 5 (ungradable) exclue automatiquement.
    """

    CLASSES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

    def __init__(self, conf, train=True, split=None):
        self.tasks    = conf.tasks
        self.img_size = conf.img_size

        if split is None:
            split = 'train' if train else 'valid'
        assert split in ('train', 'valid', 'test'), \
            f"split doit être 'train', 'valid' ou 'test', reçu: {split}"
        self.split = split

        data_root  = Path(conf.data_dir) / 'DR_grading'
        label_file = data_root / f'{split}.txt'
        img_dir    = data_root / split

        self._data = []
        with open(label_file, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                fname, grade = parts[0], int(parts[1])
                if grade == 5:
                    continue
                img_path = img_dir / fname
                if img_path.exists():
                    self._data.append((img_path, grade))

        counts = [0] * 5
        for _, g in self._data:
            counts[g] += 1
        dist_str = ', '.join(f"C{i}={counts[i]}" for i in range(5))
        print(f"[DDRDatasetMSIPS] split={split:5s} | "
              f"{len(self._data):5d} imgs | {dist_str}")

        if split == 'train':
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(
                    brightness=0.2, contrast=0.2,
                    saturation=0.1, hue=0.05),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self._data)

    def __getitem__(self, i):
        path, label = self._data[i]
        img = Image.open(path).convert('RGB')
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        img = torch.from_numpy(arr).permute(2, 0, 1)  # (3, H, W)
        img = self.transform(img)

        data_dict = {'input': img}  # (3, img_size, img_size)
        for task in self.tasks.values():
            data_dict[task['name']] = label
        return data_dict

# IV-DataFactory

In [42]:
# ============================================================
# DataFactory — IPS + MS-IPS
# ============================================================
from torch.utils.data import DataLoader


class DataFactory:
    """
    Instancie train/test datasets et dataloaders.

    Usage IPS :
        factory = DataFactory(node21_conf, dataset="node21")
        train_loader, test_loader = factory.loaders()

    Usage MS-IPS :
        factory = DataFactory(msips_node21_conf, dataset="msips_node21")
        train_loader, test_loader = factory.loaders()
    """

    _registry = {
        # IPS — retourne patches (N, C, H, W)
        #"node21":       NODE21Dataset,
        # "camelyon":   CamelyonFeatures,
        # "luna":       LUNAFeatures,
        # "lungct":     LungCTFeatures,
        # "mnist":      MegapixelMNIST,
        # "traffic":    TrafficSigns,

        # MS-IPS — retourne image entière (C, H, W)
        "msips_ddr": DDRDatasetMSIPS,
    }

    def __init__(self, conf, dataset="node21"):
        if dataset not in self._registry:
            raise ValueError(
                f"Dataset '{dataset}' inconnu.\n"
                f"Disponibles : {list(self._registry.keys())}")

        self.conf    = conf
        self.dataset = dataset
        cls          = self._registry[dataset]

        self.train_data = cls(conf, train=True)
        self.test_data  = cls(conf, train=False)

        print(f"DataFactory ✓ — dataset={dataset}")
        print(f"  Train : {len(self.train_data)} images")
        print(f"  Test  : {len(self.test_data)} images")

    def loaders(self):
        conf = self.conf
        train_loader = DataLoader(
            self.train_data,
            batch_size=conf.B_seq,
            shuffle=True,
            num_workers=conf.n_worker,
            pin_memory=conf.pin_memory,
            persistent_workers=True
        )
        test_loader = DataLoader(
            self.test_data,
            batch_size=conf.B_seq,
            shuffle=False,
            num_workers=conf.n_worker,
            pin_memory=conf.pin_memory,
            persistent_workers=True
        )
        print(f"  Loaders créés — batch_size={conf.B_seq}")
        print(f"  Train batches : {len(train_loader)}")
        print(f"  Test batches  : {len(test_loader)}")
        return train_loader, test_loader


# ── Vérification ─────────────────────────────────────────────
factory = DataFactory(msips3_ddr_up_conf, dataset="msips_ddr")
train_loader, test_loader = factory.loaders()

batch = next(iter(train_loader))
print(f"\nBatch shape : {batch['input'].shape}")
print(f"Label       : {batch['dr_grade']}")


[DDRDatasetMSIPS] split=train |  6260 imgs | C0=3133, C1=315, C2=2238, C3=118, C4=456
[DDRDatasetMSIPS] split=valid |  2503 imgs | C0=1253, C1=126, C2=895, C3=47, C4=182
DataFactory ✓ — dataset=msips_ddr
  Train : 6260 images
  Test  : 2503 images
  Loaders créés — batch_size=8
  Train batches : 783
  Test batches  : 313

Batch shape : torch.Size([8, 3, 896, 896])
Label       : tensor([0, 2, 3, 0, 4, 1, 2, 2])


# V-Architecture MS-IPS à N échelles

In [43]:
# ============================================================
# STEP 4 — Architecture MS-IPS généralisée à N échelles
#
# Différences avec la version à deux échelles :
#   - conf.scales : liste de tailles, du plus grossier au plus fin.
#                   La taille sert aussi de pas, donc chaque échelle
#                   pave l'image exactement une fois.
#   - conf.patch_out : taille du tenseur commun envoyé à l'encodeur.
#                   Toutes les échelles y sont ramenées.
#                   patch_out = max(scales) reproduit l'ancienne
#                   convention (les fines sont sur-échantillonnées).
#                   patch_out = min(scales) donne une pyramide stricte
#                   (les grossières sont sous-échantillonnées, chaque
#                   échelle apporte alors ce que les autres n'ont pas).
#
# Ordre des candidats : échelle 0 d'abord, puis 1, etc. ; à l'intérieur
# d'une échelle, parcours ligne par ligne. mem_scales donne l'indice
# d'échelle de chaque patch retenu (0 = le plus grossier).
# ============================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── 1. Encodeur de patches (inchangé) ────────────────────────

class MSIPSPatchEncoder(nn.Module):
    """Supporte resnet18 | resnet50 | efficientnet_b0."""

    def __init__(self, enc_type='resnet18', pretrained=True,
                 n_chan_in=1, n_res_blocks=4):
        super().__init__()
        self.enc_type = enc_type
        if enc_type in ('resnet18', 'resnet50'):
            self.output_dim, enc = self._build_resnet(
                enc_type, pretrained, n_chan_in, n_res_blocks)
            self.encoder = enc
        elif enc_type == 'efficientnet_b0':
            self.output_dim, enc = self._build_efficientnet(pretrained, n_chan_in)
            self.encoder = enc
        else:
            raise ValueError(f"enc_type '{enc_type}' non reconnu.")

    def _load_pretrained(self, net, pretrained, weights_true, net_fn, net_name):
        if pretrained is True:
            net = net_fn(weights=weights_true)
            print(f"[Encoder] ImageNet ({net_name})")
        elif pretrained is None or pretrained is False:
            net = net_fn(weights=None)
            print(f"[Encoder] Poids aléatoires ({net_name})")
        elif hasattr(pretrained, 'url') or hasattr(pretrained, 'value'):
            net = net_fn(weights=pretrained)
            print(f"[Encoder] Weights torchvision : {pretrained}")
        elif isinstance(pretrained, str):
            net = net_fn(weights=None)
            state = torch.load(pretrained, map_location='cpu', weights_only=True)
            if isinstance(state, dict) and 'model_state' in state:
                state = state['model_state']
            missing, unexpected = net.load_state_dict(state, strict=False)
            print(f"[Encoder] Poids depuis {pretrained}")
        else:
            raise ValueError(f"pretrained='{pretrained}' non reconnu.")
        return net

    def _build_resnet(self, enc_type, pretrained, n_chan_in, n_res_blocks):
        from torchvision.models import (resnet18, resnet50,
                                        ResNet18_Weights, ResNet50_Weights)
        if enc_type == 'resnet18':
            net_fn, weights_true, net_name = resnet18, ResNet18_Weights.IMAGENET1K_V1, 'ResNet18'
            dim_map = {2: 128, 3: 256, 4: 512}
        else:
            net_fn, weights_true, net_name = resnet50, ResNet50_Weights.IMAGENET1K_V1, 'ResNet50'
            dim_map = {2: 512, 3: 1024, 4: 2048}

        net = net_fn(weights=None)
        net = self._load_pretrained(net, pretrained, weights_true, net_fn, net_name)

        if n_chan_in != 3:
            old_conv = net.conv1
            net.conv1 = nn.Conv2d(n_chan_in, 64, kernel_size=7,
                                  stride=2, padding=3, bias=False)
            if pretrained is not None and pretrained is not False:
                net.conv1.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)

        layers = [net.conv1, net.bn1, net.relu, net.maxpool, net.layer1, net.layer2]
        if n_res_blocks >= 3:
            layers.append(net.layer3)
        if n_res_blocks >= 4:
            layers.append(net.layer4)
        layers.append(net.avgpool)
        return dim_map.get(n_res_blocks, 512), nn.Sequential(*layers)

    def _build_efficientnet(self, pretrained, n_chan_in):
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        net = efficientnet_b0(weights=None)
        net = self._load_pretrained(net, pretrained, EfficientNet_B0_Weights.IMAGENET1K_V1,
                                    efficientnet_b0, 'EfficientNet-B0')
        if n_chan_in != 3:
            old_conv = net.features[0][0]
            new_conv = nn.Conv2d(n_chan_in, old_conv.out_channels,
                                 kernel_size=old_conv.kernel_size,
                                 stride=old_conv.stride, padding=old_conv.padding,
                                 bias=old_conv.bias is not None)
            if pretrained is not None and pretrained is not False:
                with torch.no_grad():
                    new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
            net.features[0][0] = new_conv
        return 1280, nn.Sequential(net.features, net.avgpool)

    def forward(self, x):
        return torch.flatten(self.encoder(x), 1)


# ── 2. Patchifier à N échelles ───────────────────────────────

class MultiScalePatchifierND(nn.Module):
    """
    Découpe l'image à N échelles. Taille = pas, donc chaque échelle
    pave l'image exactement une fois (img_size doit être divisible
    par chaque taille).
    """

    def __init__(self, scales, img_size, n_chan, patch_out):
        super().__init__()
        self.scales = [int(s) for s in scales]
        self.img_size = img_size
        self.n_chan = n_chan
        self.patch_out = int(patch_out)

        for s in self.scales:
            assert img_size % s == 0, \
                f"img_size={img_size} n'est pas divisible par l'échelle {s}"

        self.n_cols = [img_size // s for s in self.scales]
        self.n_per_scale = [c * c for c in self.n_cols]
        self.offsets = [0]
        for n in self.n_per_scale:
            self.offsets.append(self.offsets[-1] + n)
        self.n_total = self.offsets[-1]

    def _extract(self, img, p):
        """img : (C, H, W) -> (n, C, p, p), parcours ligne par ligne."""
        patches = img.unfold(1, p, p).unfold(2, p, p)
        patches = patches.permute(1, 2, 0, 3, 4)
        return patches.reshape(-1, self.n_chan, p, p)

    def forward(self, img):
        """img : (C, H, W) -> liste de N tenseurs, un par échelle."""
        return [self._extract(img, s) for s in self.scales]

    def decode(self, idx):
        """indice global -> (échelle, ligne, colonne, taille) en pixels."""
        idx = int(idx)
        si = 0
        while si + 1 < len(self.offsets) - 1 and idx >= self.offsets[si + 1]:
            si += 1
        loc = idx - self.offsets[si]
        p, nc = self.scales[si], self.n_cols[si]
        r, c = divmod(loc, nc)
        return si, r * p, c * p, p


# ── 3. Plongement d'échelle ──────────────────────────────────

class ScaleEmbedding(nn.Module):
    def __init__(self, D, n_scales, enabled=True):
        super().__init__()
        self.enabled = enabled
        if enabled:
            self.embedding = nn.Embedding(n_scales, D)
            nn.init.normal_(self.embedding.weight, std=0.02)

    def forward(self, features, scales):
        if not self.enabled:
            return features
        emb = self.embedding(scales.to(features.device))
        return features + (emb.unsqueeze(0) if features.dim() == 3 else emb)
class ScaledDotProductAttention(nn.Module):
    def __init__(self, temperature, attn_dropout=0.1):
        super().__init__()
        self.temperature = temperature
        self.dropout     = nn.Dropout(attn_dropout)

    def compute_attn(self, q, k):
        attn = torch.matmul(q / self.temperature, k.transpose(2, 3))
        attn = self.dropout(torch.softmax(attn, dim=-1))
        return attn

    def forward(self, q, k, v):
        attn   = self.compute_attn(q, k)
        output = torch.matmul(attn, v)
        return output


class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_token, H, D, D_k, D_v,
                 attn_dropout=0.1, dropout=0.1):
        super().__init__()
        self.n_token = n_token
        self.H       = H
        self.D_k     = D_k
        self.D_v     = D_v

        self.q = nn.Parameter(torch.empty((1, n_token, D)))
        nn.init.uniform_(self.q, a=-math.sqrt(1/D_k),
                                  b= math.sqrt(1/D_k))

        self.q_w = nn.Linear(D, H * D_k, bias=False)
        self.k_w = nn.Linear(D, H * D_k, bias=False)
        self.v_w = nn.Linear(D, H * D_v, bias=False)
        self.fc  = nn.Linear(H * D_v, D, bias=False)

        self.attention  = ScaledDotProductAttention(
            temperature=D_k ** 0.5, attn_dropout=attn_dropout)
        self.dropout    = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(D, eps=1e-6)

    def get_attn(self, x):
        D_k, H, n_token = self.D_k, self.H, self.n_token
        B, len_seq      = x.shape[:2]
        q = self.q_w(self.q).view(1, n_token, H, D_k)
        k = self.k_w(x).view(B, len_seq, H, D_k)
        q, k = q.transpose(1, 2), k.transpose(1, 2)
        return self.attention.compute_attn(q, k)

    def forward(self, x):
        D_k, D_v, H, n_token = self.D_k, self.D_v, self.H, self.n_token
        B, len_seq = x.shape[:2]
        q = self.q_w(self.q).view(1, n_token, H, D_k)
        k = self.k_w(x).view(B, len_seq, H, D_k)
        v = self.v_w(x).view(B, len_seq, H, D_v)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        x = self.attention(q, k, v)
        x = x.transpose(1, 2).contiguous().view(B, n_token, -1)
        x = self.dropout(self.fc(x))
        x += self.q
        x = self.layer_norm(x)
        return x


class MLP(nn.Module):
    def __init__(self, D, D_inner, dropout=0.1):
        super().__init__()
        self.w_1        = nn.Linear(D, D_inner)
        self.w_2        = nn.Linear(D_inner, D)
        self.layer_norm = nn.LayerNorm(D, eps=1e-6)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.w_2(torch.relu(self.w_1(x)))
        x = self.dropout(x)
        x += residual
        x = self.layer_norm(x)
        return x




def pos_enc_1d(D, len_seq):
    if D % 2 != 0:
        raise ValueError(f"pos_enc_1d requiert D pair (reçu {D})")
    pe       = torch.zeros(len_seq, D)
    position = torch.arange(0, len_seq).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, D, 2, dtype=torch.float) *
                         -(math.log(10000.0) / D))
    pe[:, 0::2] = torch.sin(position.float() * div_term)
    pe[:, 1::2] = torch.cos(position.float() * div_term)
    return pe


class MSTransformer(nn.Module):
    """
    Identique au Transformer IPS original.
    Les features reçues ont déjà le scale embedding injecté.
    """
    def __init__(self, n_token, H, D, D_k, D_v,
                 D_inner, attn_dropout=0.1, dropout=0.1):
        super().__init__()
        self.crs_attn = MultiHeadCrossAttention(
            n_token, H, D, D_k, D_v,
            attn_dropout=attn_dropout, dropout=dropout)
        self.mlp = MLP(D, D_inner, dropout=dropout)

    def get_scores(self, x):
        attn = self.crs_attn.get_attn(x)
        return attn.mean(dim=1).transpose(1, 2).mean(-1)

    def forward(self, x):
        return self.mlp(self.crs_attn(x))


# ── 5. MSIPSNet ──────────────────────────────────────────────



# ── 4bis. Encodage positionnel (repris tel quel) ─────────────

def pos_enc_1d(D, len_seq):
    if D % 2 != 0:
        raise ValueError(f"pos_enc_1d requiert D pair (reçu {D})")
    pe       = torch.zeros(len_seq, D)
    position = torch.arange(0, len_seq).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, D, 2, dtype=torch.float) *
                         -(math.log(10000.0) / D))
    pe[:, 0::2] = torch.sin(position.float() * div_term)
    pe[:, 1::2] = torch.cos(position.float() * div_term)
    return pe


# ── 5. MSIPSNet à N échelles ─────────────────────────────────

class MSIPSNet(nn.Module):
    """
    Pipeline IPS :
      image -> patchifier N échelles -> toutes ramenées à patch_out
            -> encodeur (sans gradient) -> projecteur -> plongement d'échelle
            -> IPS itératif -> Top-M

    Pipeline forward (avec gradient) :
      mem_patches (M, C, patch_out, patch_out) + mem_scales (M,)
            -> encodeur -> projecteur -> plongement -> transformer -> tête
    """

    def __init__(self, device, conf):
        super().__init__()
        self.device        = device
        self.M             = conf.M
        self.I             = conf.I
        self.D             = conf.D
        self.tasks         = conf.tasks
        self.n_class       = conf.n_class
        self.use_pos       = conf.use_pos
        self.scales        = [int(s) for s in conf.scales]
        self.patch_out     = int(getattr(conf, 'patch_out', max(conf.scales)))
        self.use_scale_emb = getattr(conf, 'use_scale_emb', True)

        self.use_checkpoint   = getattr(conf, 'use_checkpoint',   False)
        self.ips_encode_batch = getattr(conf, 'ips_encode_batch', None)

        self.patchifier = MultiScalePatchifierND(
            scales=self.scales, img_size=conf.img_size,
            n_chan=conf.n_chan_in, patch_out=self.patch_out)

        self.encoder = MSIPSPatchEncoder(
            enc_type=getattr(conf, 'enc_type', 'resnet18'),
            pretrained=conf.pretrained,
            n_chan_in=conf.n_chan_in,
            n_res_blocks=getattr(conf, 'n_res_blocks', 4))
        enc_dim = self.encoder.output_dim

        self.projector = nn.Sequential(
            nn.LayerNorm(enc_dim, elementwise_affine=False),
            nn.Linear(enc_dim, conf.D),
            nn.BatchNorm1d(conf.D),
            nn.ReLU())

        self.scale_emb = ScaleEmbedding(
            conf.D, n_scales=len(self.scales), enabled=self.use_scale_emb)

        if self.use_pos:
            self.pos_enc = pos_enc_1d(
                conf.D, self.patchifier.n_total).unsqueeze(0).to(device)
        else:
            self.pos_enc = None

        self.transf = MSTransformer(
            conf.n_token, conf.H, conf.D, conf.D_k, conf.D_v,
            conf.D_inner, conf.attn_dropout, conf.dropout)

        self.output_layers = nn.ModuleDict()
        for task in conf.tasks.values():
            act = (nn.Softmax(dim=-1) if task['act_fn'] == 'softmax' else nn.Sigmoid())
            self.output_layers[task['name']] = nn.Sequential(
                nn.Linear(conf.D, conf.n_class), act)

        # vecteur d'échelle de chaque candidat, calculé une fois
        sc = torch.cat([torch.full((n,), si, dtype=torch.long)
                        for si, n in enumerate(self.patchifier.n_per_scale)])
        self.register_buffer('scales_all', sc, persistent=False)

    @torch.no_grad()
    def ips(self, imgs):
        """
        imgs : (B, C, H, W)
        retourne mem_patches (B, M, C, P, P), mem_scales (B, M),
                 mem_idx (B, M), mem_pos (B, M, D) ou None
        """
        M, I, D = self.M, self.I, self.D
        device  = self.device
        P       = self.patch_out
        B       = imgs.shape[0]

        if self.training:
            self.encoder.eval(); self.transf.eval()
            self.projector.eval(); self.scale_emb.eval()

        # 1. découpe par échelle, puis mise à la taille commune
        per_scale = [[] for _ in self.scales]
        for b in range(B):
            for si, t in enumerate(self.patchifier(imgs[b])):
                per_scale[si].append(t)

        blocks = []
        for si, lst in enumerate(per_scale):
            t = torch.stack(lst, dim=0)                 # (B, n_si, C, s, s)
            n_si = t.shape[1]
            if self.scales[si] != P:
                t = F.interpolate(
                    t.reshape(B * n_si, *t.shape[2:]),
                    size=(P, P), mode='bilinear', align_corners=False
                ).reshape(B, n_si, t.shape[2], P, P)
            blocks.append(t)

        all_patches = torch.cat(blocks, dim=1).to(device)   # (B, N, C, P, P)
        N = all_patches.shape[1]
        assert N == self.patchifier.n_total

        # 2. encodage de tous les candidats, sans gradient
        shp  = all_patches.shape
        flat = all_patches.reshape(B * N, *shp[2:])
        if self.ips_encode_batch is None:
            feats = self.encoder(flat)
        else:
            feats = torch.cat([self.encoder(flat[i:i + self.ips_encode_batch])
                               for i in range(0, len(flat), self.ips_encode_batch)], dim=0)
        feats = self.projector(feats)

        sc_all  = self.scales_all.to(device)
        sc_flat = sc_all.unsqueeze(0).expand(B, -1).reshape(B * N)
        feats   = self.scale_emb(feats, sc_flat)
        feats_b = feats.view(B, N, D)

        pos = self.pos_enc.expand(B, -1, -1) if (self.use_pos and self.pos_enc is not None) else None

        # 3. IPS itératif
        idx_all = torch.arange(N, dtype=torch.int64, device=device).unsqueeze(0).expand(B, -1)
        mem_emb = feats_b[:, :M]
        mem_idx = idx_all[:, :M]

        for i in range(math.ceil((N - M) / I)):
            a = i * I + M
            b_ = min(a + I, N)
            cat_emb = torch.cat([mem_emb, feats_b[:, a:b_]], dim=1)
            cat_idx = torch.cat([mem_idx, idx_all[:, a:b_]], dim=1)
            if pos is not None:
                score_emb = cat_emb + torch.gather(
                    pos, 1, cat_idx.unsqueeze(-1).expand(-1, -1, D))
            else:
                score_emb = cat_emb
            top = torch.topk(self.transf.get_scores(score_emb), M, dim=-1)[1].squeeze(1)
            mem_idx = torch.gather(cat_idx, 1, top)
            mem_emb = torch.gather(feats_b, 1, mem_idx.unsqueeze(-1).expand(-1, -1, D))

        # 4. récupération des patchs retenus
        mem_patches = torch.gather(
            all_patches, 1,
            mem_idx.view(B, -1, 1, 1, 1).expand(-1, -1, *shp[2:]))
        mem_scales = sc_all[mem_idx]
        mem_pos = (torch.gather(pos, 1, mem_idx.unsqueeze(-1).expand(-1, -1, D))
                   if pos is not None else None)

        if self.training:
            self.encoder.train(); self.transf.train()
            self.projector.train(); self.scale_emb.train()

        return mem_patches, mem_scales, mem_idx, mem_pos

    def forward(self, mem_patches, mem_scales, mem_pos=None):
        B, M = mem_patches.shape[:2]
        flat_patches = mem_patches.reshape(B * M, *mem_patches.shape[2:])
        flat_scales  = mem_scales.reshape(B * M)

        if self.use_checkpoint:
            from torch.utils.checkpoint import checkpoint as ckpt
            feats = ckpt(self.encoder, flat_patches, use_reentrant=False)
        else:
            feats = self.encoder(flat_patches)
        feats = self.projector(feats)
        feats = self.scale_emb(feats, flat_scales)
        feats_b = feats.view(B, M, self.D)
        if mem_pos is not None:
            feats_b = feats_b + mem_pos

        image_emb = self.transf(feats_b)
        return {t['name']: self.output_layers[t['name']](image_emb[:, t['id']])
                for t in self.tasks.values()}


# VI-Utils

In [44]:
# ============================================================
# Cellule 5a — Logger + fonctions utilitaires
# Copié depuis utils/utils.py — fond inchangé
# ============================================================
import math
import numpy as np
from collections import defaultdict
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              f1_score, precision_score, recall_score)
import torch
import torch.nn as nn


def adjust_learning_rate(n_epoch_warmup, n_epoch, max_lr,
                         optimizer, dloader, step):
    """Ajuste le learning rate selon un schedule cosinus avec warmup."""
    max_steps    = int(n_epoch * len(dloader))
    warmup_steps = int(n_epoch_warmup * len(dloader))

    if step < warmup_steps:
        lr = max_lr * step / warmup_steps
    else:
        step      -= warmup_steps
        max_steps -= warmup_steps
        q          = 0.5 * (1 + math.cos(math.pi * step / max_steps))
        end_lr     = max_lr * 0.001
        lr         = max_lr * q + end_lr * (1 - q)

    optimizer.param_groups[0]['lr'] = lr


def shuffle_batch(x, shuffle_idx=None):
    """Mélange chaque instance du batch de la même façon."""
    if not torch.is_tensor(shuffle_idx):
        seq_len     = x.shape[1]
        shuffle_idx = torch.randperm(seq_len)
    x = x[:, shuffle_idx]
    return x, shuffle_idx


def shuffle_instance(x, axis, shuffle_idx=None):
    """Mélange chaque instance du batch différemment."""
    if not torch.is_tensor(shuffle_idx):
        shuffle_idx = torch.rand(
            x.shape[:axis+1], device=x.device).argsort(axis)

    idx_expand = shuffle_idx.clone().to(x.device)
    for _ in range(x.ndim - axis - 1):
        idx_expand.unsqueeze_(-1)
    idx_expand = idx_expand.repeat(
        *[1 for _ in range(axis+1)], *(x.shape[axis+1:]))

    x = x.gather(axis, idx_expand)
    return x, shuffle_idx


class Logger(nn.Module):
    """Stocke et calcule les statistiques de pertes et métriques."""

    def __init__(self, task_dict):
        super().__init__()
        self.task_dict    = task_dict
        self.losses_it    = defaultdict(list)
        self.losses_epoch = defaultdict(list)
        self.y_preds      = defaultdict(list)
        self.y_trues      = defaultdict(list)
        self.metrics      = defaultdict(list)

    def update(self, next_loss, next_y_pred, next_y_true):

        for task in self.task_dict.values():
            t      = task['name']
            t_metr = task['metric']
            if isinstance(t_metr, str):
                t_metr = [t_metr]

            self.losses_it[t].append(next_loss[t])

            has_auc_f1 = any(m in ['auc', 'f1', 'precision', 'recall']
                             for m in t_metr)

            if t_metr == ['accuracy']:
                y_pred = np.argmax(next_y_pred[t], axis=-1)
            elif has_auc_f1:
                raw = next_y_pred[t]
                if raw.ndim == 2:
                    y_pred = raw[:, 1].tolist()
                else:
                    y_pred = raw.tolist()
            else:
                y_pred = next_y_pred[t].tolist()

            self.y_preds[t].extend(y_pred)
            self.y_trues[t].extend(next_y_true[t])

    def compute_metric(self):

        for task in self.task_dict.values():
            t      = task['name']
            losses = self.losses_it[t]
            self.losses_epoch[t].append(np.mean(losses))

            requested = task['metric']
            if isinstance(requested, str):
                requested = [requested]

            y_true     = np.array(self.y_trues[t])
            y_pred_raw = np.array(self.y_preds[t])

            epoch_metrics = {}
            for current_metric in requested:

                if current_metric == 'accuracy':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    epoch_metrics[current_metric] = accuracy_score(
                        y_true, y_pred)

                elif current_metric == 'multilabel_accuracy':
                    y_pred  = np.where(y_pred_raw >= 0.5, 1., 0.)
                    correct = np.all(y_pred == y_true, axis=-1).sum()
                    epoch_metrics[current_metric] = correct / len(y_true)

                elif current_metric == 'auc':
                    scores = (y_pred_raw[:, 1]
                              if y_pred_raw.ndim == 2 else y_pred_raw)
                    epoch_metrics[current_metric] = roc_auc_score(
                        y_true, scores)

                elif current_metric == 'f1':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    epoch_metrics[current_metric] = f1_score(
                        y_true, y_pred, zero_division=0)

                elif current_metric == 'precision':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    epoch_metrics[current_metric] = precision_score(
                        y_true, y_pred, zero_division=0)

                elif current_metric == 'recall':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    epoch_metrics[current_metric] = recall_score(
                        y_true, y_pred, zero_division=0)

            self.metrics[t].append(epoch_metrics)

            # Réinitialisation pour la prochaine époque
            self.losses_it[t] = []
            self.y_preds[t]   = []
            self.y_trues[t]   = []

    def print_stats(self, epoch, train,epoch_real=None, **kwargs):
        
        print_str  = 'Train' if train else 'Test'
        display    = (epoch_real + 1) if epoch_real is not None else (epoch + 1)
        print_str += f" Epoch: {display}\n"

        avg_loss = 0
        for task in self.task_dict.values():
            t         = task['name']
            mean_loss = self.losses_epoch[t][epoch]
            avg_loss += mean_loss

            epoch_metrics = self.metrics[t][epoch]
            metrics_str   = ', '.join(
                f"{k}: {v:.5f}" for k, v in epoch_metrics.items())
            print_str += f"task: {t}, loss: {mean_loss:.5f}, {metrics_str}\n"

        avg_loss  /= len(self.task_dict.values())
        print_str += f"avg loss: {avg_loss:.5f}"

        for k, v in kwargs.items():
            print_str += f", {k}: {v}"
        print_str += "\n"

        print(print_str)



def plot_histories(histories, metric="loss", figsize=(12, 5)):
    """
    Trace les courbes train/val pour une métrique donnée,
    sur plusieurs runs simultanément.

    Args:
        histories : dict {run_name: history_dict}
                    — chargé via load_history() ou depuis engine.history
        metric    : "loss", "accuracy", "auc", "f1", etc.
        figsize   : taille figure

    Usage :
        # Charger plusieurs runs
        histories = {}
        for name, path in [
            ("ips_node21_v1",  "checkpoints/ips_node21_v1_history.json"),
            ("ips_node21_v2",  "checkpoints/ips_node21_v2_history.json"),
        ]:
            _, h = UniversalEngine.load_history(path)
            histories[name] = h

        # Tracer
        plot_histories(histories, metric="auc")
        plot_histories(histories, metric="loss")
    """
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import numpy as np

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    colors    = plt.cm.tab10(np.linspace(0, 1, len(histories)))

    for ax, phase in zip(axes, ["train", "val"]):
        for (run_name, history), color in zip(histories.items(), colors):
            vals = history[phase].get(metric, [])
            if not vals:
                continue
            epochs = range(1, len(vals) + 1)
            ax.plot(epochs, vals,
                    label=run_name, color=color,
                    linewidth=2, marker='o', markersize=4)

            # Annoter le meilleur point
            if phase == "val":
                best_fn = min if metric == "loss" else max
                best    = best_fn(vals)
                ep_best = vals.index(best) + 1
                ax.annotate(
                    f"{best:.4f}",
                    xy=(ep_best, best),
                    xytext=(5, 5), textcoords="offset points",
                    fontsize=8, color=color)

        ax.set_title(f"{metric} — {phase}", fontsize=12)
        ax.set_xlabel("Époque")
        ax.set_ylabel(metric)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle(f"Comparaison runs — {metric}", fontsize=13)
    plt.tight_layout()
    plt.show()


def plot_all_metrics(histories, figsize=(16, 10)):
    """
    Grille complète : une ligne par métrique (loss + toutes métriques),
    deux colonnes (train / val).

    Usage :
        plot_all_metrics(histories)
    """
    import matplotlib.pyplot as plt
    import numpy as np

    # Collecter toutes les métriques disponibles
    first_history = next(iter(histories.values()))
    metrics       = list(first_history["val"].keys())  # loss, accuracy, auc...

    n_rows  = len(metrics)
    fig, axes = plt.subplots(n_rows, 2,
                             figsize=(figsize[0], figsize[1] * n_rows // 2))
    if n_rows == 1:
        axes = [axes]   # forcer 2D

    colors = plt.cm.tab10(np.linspace(0, 1, len(histories)))

    for row_idx, metric in enumerate(metrics):
        for col_idx, phase in enumerate(["train", "val"]):
            ax = axes[row_idx][col_idx]

            for (run_name, history), color in zip(
                    histories.items(), colors):
                vals = history[phase].get(metric, [])
                if not vals:
                    continue
                epochs = range(1, len(vals) + 1)
                ax.plot(epochs, vals,
                        label=run_name, color=color,
                        linewidth=2, marker='o', markersize=3)

            ax.set_title(f"{metric} [{phase}]", fontsize=10)
            ax.set_xlabel("Époque")
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)

    plt.suptitle("Toutes métriques — comparaison runs", fontsize=13)
    plt.tight_layout()
    plt.show()


# VRAM compute

def get_total_vram_mb():
    """Somme la VRAM peak sur tous les GPUs disponibles."""
    if not torch.cuda.is_available():
        return 0.0
    total = 0.0
    for i in range(torch.cuda.device_count()):
        total += torch.cuda.max_memory_allocated(device=i)
    return total / 1e6

# ── Vérification ────────────────────────────────────────────
log = Logger(msips3_ddr_up_conf.tasks)
print("Logger instancié ✓")
print(f"Tasks : {[t['name'] for t in msips3_ddr_up_conf.tasks.values()]}")
print(f"adjust_learning_rate : ✓")
print(f"shuffle_batch        : ✓")
print(f"shuffle_instance     : ✓")

Logger instancié ✓
Tasks : ['dr_grade']
adjust_learning_rate : ✓
shuffle_batch        : ✓
shuffle_instance     : ✓


# VII-UniversalEngine

In [45]:
# ============================================================
# STEP 5b — UniversalEngine MS-IPS
# Différences vs IPS :
#   - _init_batch    : buffer pour patches + scales (pas pos_enc)
#   - _fill_batch    : stocke mem_patches + mem_scales
#   - _compute_loss  : net(mem_patches, mem_scales)
#   - _train/_eval   : net.ips(img) retourne (patches, scales, idx)
# Le reste est identique à ips.ipynb
# ============================================================
import os, csv, json, time, math
import torch
import torch.nn as nn
import numpy as np


class UniversalEngineMSIPS:

    def __init__(self, net, conf, device, run_name="msips_run"):
        self.net      = net
        self.conf     = conf
        self.device   = device
        self.run_name = run_name

        if torch.cuda.device_count() > 1:
            print(f"DataParallel activé : {torch.cuda.device_count()} GPUs")
            self.net = nn.DataParallel(self.net)
        self.net.to(device)

        self.optimizer = torch.optim.AdamW(
            self.net.parameters(), lr=0, weight_decay=conf.wd)

        loss_nll = nn.NLLLoss()
        loss_bce = nn.BCELoss()
        self.criterions = {
            task['name']: loss_nll if task['act_fn'] == 'softmax'
                          else loss_bce
            for task in conf.tasks.values()
        }

        metric_names = []
        for task in conf.tasks.values():
            requested = task['metric']
            if isinstance(requested, str):
                requested = [requested]
            metric_names.extend(requested)
        self._metric_names = metric_names

        self.history = {
            "train": {"loss": [], **{m: [] for m in metric_names}},
            "val":   {"loss": [], **{m: [] for m in metric_names}},
            "lr":    [],
            "efficiency": {}
        }
        self.start_epoch = 0

        print(f"UniversalEngineMSIPS instancié ✓")
        print(f"Optimizer : AdamW")
        print(f"Critères  : {list(self.criterions.keys())}")
        print(f"History keys (train) : "
              f"{list(self.history['train'].keys())}")

    # ── Internes ─────────────────────────────────────────────

    def _init_batch(self):
        conf   = self.conf
        device = self.device
        M, C, H = conf.M, conf.n_chan_in, conf.patch_out
    
        mem_patches = torch.zeros((conf.B, M, C, H, H)).to(device)
        mem_scales  = torch.zeros((conf.B, M), dtype=torch.long).to(device)
        mem_pos     = (torch.zeros((conf.B, M, conf.D)).to(device)
                       if conf.use_pos else None)
    
        labels = {}
        for task in conf.tasks.values():
            labels[task['name']] = torch.zeros(
                (conf.B,), dtype=torch.int64).to(device)
    
        return mem_patches, mem_scales, mem_pos, labels
    
    
    def _fill_batch(self, mem_patches, mem_scales, mem_pos, labels,
                    data, n_prep, n_prep_batch,
                    patches_iter, scales_iter, pos_iter):
        """
        patches_iter : (B_seq, M, C, H, W)
        scales_iter  : (B_seq, M)
        pos_iter     : (B_seq, M, D) ou None
        """
        B_seq = patches_iter.shape[0]
        end   = n_prep + B_seq
    
        mem_patches[n_prep:end] = patches_iter
        mem_scales[n_prep:end]  = scales_iter
        if mem_pos is not None and pos_iter is not None:
            mem_pos[n_prep:end] = pos_iter
    
        for task in self.conf.tasks.values():
            labels[task['name']][n_prep:end] = data[task['name']].to(
                labels[task['name']].device)
    
        return (mem_patches, mem_scales, mem_pos, labels,
                end, n_prep_batch + 1)
    
    
    def _shrink_batch(self, mem_patches, mem_scales, mem_pos, labels, n_prep):
        mem_patches = mem_patches[:n_prep]
        mem_scales  = mem_scales[:n_prep]
        if mem_pos is not None:
            mem_pos = mem_pos[:n_prep]
        for task in self.conf.tasks.values():
            labels[task['name']] = labels[task['name']][:n_prep]
        return mem_patches, mem_scales, mem_pos, labels
    
    
    def _compute_loss(self, mem_patches, mem_scales, mem_pos, labels):
        conf = self.conf
        net  = (self.net.module
                if isinstance(self.net, nn.DataParallel) else self.net)
    
        preds = net(mem_patches, mem_scales, mem_pos)
    
        loss        = 0
        task_losses = {}
        task_preds  = {}
        task_labels = {}
    
        for task in conf.tasks.values():
            t_name = task['name']
            pred   = preds[t_name].squeeze(-1)
            label  = labels[t_name]
    
            if task['act_fn'] == 'softmax':
                pred_loss  = torch.log(pred + conf.eps)
                label_loss = label
            else:
                pred_loss  = pred.view(-1)
                label_loss = label.float().view(-1)
    
            t_loss              = self.criterions[t_name](pred_loss, label_loss)
            task_losses[t_name] = t_loss.item()
            task_preds[t_name]  = pred.detach().cpu().numpy()
            task_labels[t_name] = label.detach().cpu().numpy()
            loss               += t_loss
    
        loss /= len(conf.tasks)
        return loss, [task_losses, task_preds, task_labels]

    # ── Train one epoch ──────────────────────────────────────

    def _train_one_epoch(self, train_loader, log_writer, epoch):
        conf   = self.conf
        device = self.device
        net    = self.net
        net.train()
    
        track_epochs = getattr(conf, 'track_epochs', [0, 1])
        if isinstance(track_epochs, bool):
            track_epochs = [0, 1] if track_epochs else []
        track = epoch in track_epochs
    
        n_prep, n_prep_batch = 0, 0
        start_new_batch      = True
    
        if track:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
    
        for data_it, data in enumerate(
                train_loader, start=epoch * len(train_loader)):
                                               
            imgs  = data['input'].to(device)  # (B_seq, C, H, W)
            
            if start_new_batch:
                mem_patches, mem_scales, mem_pos, labels = self._init_batch()
                start_new_batch = False
            
            _net = (net.module
                    if isinstance(net, nn.DataParallel) else net)
            patches_iter, scales_iter, _, pos_iter = _net.ips(imgs)
            
            (mem_patches, mem_scales, mem_pos, labels,
             n_prep, n_prep_batch) = self._fill_batch(
                mem_patches, mem_scales, mem_pos, labels, data,
                n_prep, n_prep_batch, patches_iter, scales_iter, pos_iter)
            
            batch_full    = (n_prep == conf.B)
            is_last_batch = (n_prep_batch == len(train_loader))
                
            if batch_full or is_last_batch:
                if not batch_full:
                    mem_patches, mem_scales, mem_pos, labels = \
                        self._shrink_batch(
                            mem_patches, mem_scales, mem_pos, labels, n_prep)
    
                adjust_learning_rate(
                    conf.n_epoch_warmup, conf.n_epoch,
                    conf.lr, self.optimizer,
                    train_loader, data_it + 1)
    
                self.optimizer.zero_grad()
                loss, task_info = self._compute_loss(
                    mem_patches, mem_scales, mem_pos, labels)
                task_losses, task_preds, task_labels = task_info
                loss.backward()
                self.optimizer.step()
                log_writer.update(task_losses, task_preds, task_labels)
                n_prep = 0; start_new_batch = True
    
        if track:
            elapsed = time.time() - t0
            vram_mb = get_total_vram_mb()
            if epoch not in self.history["efficiency"]:
                self.history["efficiency"][epoch] = {}
            self.history["efficiency"][epoch]["train"] = {
                "vram_mb": round(vram_mb, 1),
                "time_s":  round(elapsed, 1)
            }
    # ── Évaluation ───────────────────────────────────────────

    @torch.no_grad()
    def evaluate(self, loader, epoch=None):
        conf   = self.conf
        device = self.device
        net    = self.net
        net.eval()
    
        track_epochs = getattr(conf, 'track_epochs', [0, 1])
        if isinstance(track_epochs, bool):
            track_epochs = [0, 1] if track_epochs else []
        track = (epoch is not None) and (epoch in track_epochs)
    
        log_writer      = Logger(conf.tasks)
        n_prep, n_prep_batch = 0, 0
        start_new_batch = True
    
        if track:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
    
        for data in loader:
            imgs  = data['input'].to(device)  # (B_seq, C, H, W)
            
            if start_new_batch:
                mem_patches, mem_scales, mem_pos, labels = self._init_batch()
                start_new_batch = False
            
            _net = (net.module
                    if isinstance(net, nn.DataParallel) else net)
            patches_iter, scales_iter, _, pos_iter = _net.ips(imgs)
            
            (mem_patches, mem_scales, mem_pos, labels,
             n_prep, n_prep_batch) = self._fill_batch(
                mem_patches, mem_scales, mem_pos, labels, data,
                n_prep, n_prep_batch, patches_iter, scales_iter, pos_iter)
            
            batch_full    = (n_prep == conf.B)
            is_last_batch = (n_prep_batch == len(loader))
    
            if batch_full or is_last_batch:
                if not batch_full:
                    mem_patches, mem_scales, mem_pos, labels = \
                        self._shrink_batch(
                            mem_patches, mem_scales, mem_pos, labels, n_prep)
                _, task_info = self._compute_loss(
                    mem_patches, mem_scales, mem_pos, labels)
                task_losses, task_preds, task_labels = task_info
                log_writer.update(task_losses, task_preds, task_labels)
                n_prep = 0; start_new_batch = True
    
        if track:
            elapsed = time.time() - t0
            vram_mb = get_total_vram_mb()
            if epoch not in self.history["efficiency"]:
                self.history["efficiency"][epoch] = {}
            self.history["efficiency"][epoch]["val"] = {
                "vram_mb": round(vram_mb, 1),
                "time_s":  round(elapsed, 1)
            }
    
        return log_writer
    # ── Boucle complète ──────────────────────────────────────

    # ── Checkpoint ───────────────────────────────────────────

    def save(self, path):
        os.makedirs(os.path.dirname(os.path.abspath(path)),
                    exist_ok=True)
        net_state = (self.net.module.state_dict()
                     if isinstance(self.net, nn.DataParallel)
                     else self.net.state_dict())

        def _to_native(obj):
            if isinstance(obj, dict):
                return {k: _to_native(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [_to_native(v) for v in obj]
            if hasattr(obj, 'item'):
                return obj.item()
            return obj

        torch.save({
            "epoch":           len(self.history["train"]["loss"]),
            "model_state":     net_state,
            "optimizer_state": self.optimizer.state_dict(),
            "history":         _to_native(self.history),
            "run_name":        self.run_name
        }, path)
        print(f"Checkpoint sauvegardé : {path}")

    def save_weights(self, path):
        net_state = (self.net.module.state_dict()
                     if isinstance(self.net, nn.DataParallel)
                     else self.net.state_dict())
        torch.save(net_state, path)
        print(f"Poids sauvegardés : {path}")

    def load(self, path):
        ckpt = torch.load(
            path, map_location=self.device, weights_only=False)
        net  = (self.net.module
                if isinstance(self.net, nn.DataParallel) else self.net)
        net.load_state_dict(ckpt["model_state"])
        self.optimizer.load_state_dict(ckpt["optimizer_state"])
        self.history     = ckpt["history"]
        self.start_epoch = ckpt["epoch"]
        print(f"Checkpoint chargé : {path} "
              f"(reprise époque {self.start_epoch})")

    def load_weights(self, path):
        state = torch.load(
            path, map_location=self.device, weights_only=True)
        net   = (self.net.module
                 if isinstance(self.net, nn.DataParallel) else self.net)
        net.load_state_dict(state)
        print(f"Poids chargés : {path}")

    def save_history(self, path=None):
        if path is None:
            path = f"/kaggle/working/{self.run_name}_history.json"
        def _to_native(obj):
            if isinstance(obj, dict):
                return {k: _to_native(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [_to_native(v) for v in obj]
            if hasattr(obj, 'item'):
                return obj.item()
            return obj
        with open(path, "w") as f:
            json.dump({"run_name": self.run_name,
                       "history":  _to_native(self.history)}, f, indent=2)
        print(f"History sauvegardée : {path}")

    # ── Résumé ───────────────────────────────────────────────

    def summary(self):
        net = (self.net.module
               if isinstance(self.net, nn.DataParallel) else self.net)
        n_params = sum(p.numel() for p in net.parameters()) / 1e6
        print(f"\n{'='*50}")
        print(f"Run        : {self.run_name}")
        print(f"Paramètres : {n_params:.3f} M")
        if self.history["val"]["loss"]:
            print(f"\n── Meilleurs scores (val) ──")
            best_loss = min(self.history['val']['loss'])
            ep_loss   = self.history['val']['loss'].index(best_loss)
            print(f"  {'loss':12s} : {best_loss:.5f} "
                  f"(époque {ep_loss+1})")
            for m in self._metric_names:
                vals = self.history["val"].get(m, [])
                if not vals: continue
                best = max(vals)
                ep   = vals.index(best)
                print(f"  {m:12s} : {best:.5f} (époque {ep+1})")
        if self.history["efficiency"]:
            print(f"\n── Efficiency ──")
            for ep, eff in sorted(
                    self.history["efficiency"].items()):
                for phase, stats in eff.items():
                    print(f"  Époque {ep} [{phase}] : "
                          f"VRAM={stats['vram_mb']:.0f} MB, "
                          f"temps={stats['time_s']:.1f} s")
        print(f"{'='*50}\n")

   

    def predict(self, img, return_patches=True):
        """
        Inférence sur une image entière.
    
        Args:
            img           : tensor (C, H, W) ou (1, C, H, W)
            return_patches: si True, retourne patches + scales + scores
    
        Returns:
            result : dict {
                'probs'    : {task_name: np.array (n_class,)}
                'label'    : {task_name: int}
                'patches'  : np.array (M, C, H, W)
                'scales'   : np.array (M,)
                'scores'   : np.array (M,)
                'grid_idx' : np.array (M,)
            }
        """
        conf   = self.conf
        device = self.device
        net    = (self.net.module
                  if isinstance(self.net, nn.DataParallel) else self.net)
        net.eval()
    
        if img.dim() == 3:
            img = img.unsqueeze(0)  # (1, C, H, W)
    
        with torch.no_grad():
            img_dev = img.to(device)
    
            # ── IPS ─────────────────────────────────────────────
            mem_patches, mem_scales, mem_idx, mem_pos = net.ips(img_dev)
            # mem_patches : (B, M, C, H, W)
            # mem_scales  : (B, M)
    
            # ── Scores d'attention ───────────────────────────────
            B_pred, M_pred = mem_patches.shape[:2]
            flat_patches   = mem_patches.view(B_pred * M_pred, *mem_patches.shape[2:])
            flat_scales    = mem_scales.view(B_pred * M_pred)
    
            feats   = net.encoder(flat_patches)             # (B*M, enc_dim)
            feats   = net.projector(feats)                  # (B*M, D)
            feats   = net.scale_emb(feats, flat_scales)     # (B*M, D)
            feats_b = feats.view(B_pred, M_pred, net.D)     # (B, M, D)
    
            if mem_pos is not None:
                feats_b = feats_b + mem_pos
    
            attn_scores = net.transf.get_scores(
                feats_b).squeeze(0).cpu().numpy()           # (M,)
    
            # ── Forward ──────────────────────────────────────────
            preds = net(mem_patches, mem_scales, mem_pos)
    
            result = {'probs': {}, 'label': {}}
            for task in conf.tasks.values():
                t_name = task['name']
                prob   = preds[t_name].squeeze(0).cpu().numpy()
                result['probs'][t_name] = prob
                if task['act_fn'] == 'softmax':
                    result['label'][t_name] = int(np.argmax(prob))
                else:
                    result['label'][t_name] = int(prob >= 0.5)
    
            if return_patches:
                result['patches']  = mem_patches.squeeze(0).cpu().numpy()  # (M, C, H, W)
                result['scales']   = mem_scales.squeeze(0).cpu().numpy()   # (M,)
                result['scores']   = attn_scores                           # (M,)
                result['grid_idx'] = mem_idx.squeeze(0).cpu().numpy()      # (M,)
    
        return result
            
    



    @staticmethod
    def load_history(path):
        """
        Charge un fichier history JSON.
        Retourne (run_name, history_dict).
        Méthode statique — pas besoin d'instancier l'engine.
        """
        import json
        with open(path, "r") as f:
            payload = json.load(f)
        print(f"History chargée : {path} "
              f"({payload['history']['train']['loss'].__len__()} époques)")
        return payload["run_name"], payload["history"]


# ── Vérification ─────────────────────────────────────────────
net_check = MSIPSNet(DEVICE, msips3_ddr_up_conf)
engine    = UniversalEngineMSIPS(
    net_check, msips3_ddr_up_conf, DEVICE,
    run_name="msips3_ddr_up_v1")
print(f"History keys (train) : "
      f"{list(engine.history['train'].keys())}")


[Encoder] ImageNet (ResNet18)
DataParallel activé : 2 GPUs
UniversalEngineMSIPS instancié ✓
Optimizer : AdamW
Critères  : ['dr_grade']
History keys (train) : ['loss', 'accuracy', 'kappa', 'f1']
History keys (train) : ['loss', 'accuracy', 'kappa', 'f1']


# VIII-Saillance

In [46]:
# ============================================================
# Saillance — version N échelles
#
# Remplace la version à deux échelles, qui codait en dur
# coarse_* / fine_*. Ici la géométrie est dérivée de conf.scales,
# donc le code vaut pour deux échelles comme pour trois.
#
# Attache show_saliency et save_saliency à UniversalEngineMSIPS.
# ============================================================
import os, csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Couleur de contour par échelle, du plus grossier au plus fin
_SCALE_COLORS = ['cyan', 'lime', 'magenta', 'yellow', 'orange']


def _scale_geometry(conf):
    """Retourne sizes, n_cols, offsets pour décoder un indice global."""
    S     = conf.img_size
    sizes = [int(s) for s in conf.scales]
    ncols = [S // s for s in sizes]
    offs  = [0]
    for c in ncols:
        offs.append(offs[-1] + c * c)
    return sizes, ncols, offs


def _decode(conf, idx, si):
    """indice global + indice d'échelle -> (ligne, colonne, taille) en pixels."""
    sizes, ncols, offs = _scale_geometry(conf)
    si, idx = int(si), int(idx)
    p, nc   = sizes[si], ncols[si]
    r, c    = divmod(idx - offs[si], nc)
    return r * p, c * p, p


def _score_range(scores):
    s = np.asarray(scores, dtype=float)
    lo, hi = float(s.min()), float(s.max())
    if hi - lo < 1e-12:
        hi = lo + 1e-6
    return lo, hi


def _build_heatmap(scores, grid_idx, scales_idx, conf):
    """Carte de chaleur : chaque patch retenu peint son score normalisé."""
    S = conf.img_size
    hm = np.zeros((S, S), dtype=np.float32)
    lo, hi = _score_range(scores)
    for i, (idx, si) in enumerate(zip(grid_idx, scales_idx)):
        r, c, p = _decode(conf, idx, si)
        v = (float(scores[i]) - lo) / (hi - lo)
        hm[r:r + p, c:c + p] = np.maximum(hm[r:r + p, c:c + p], v)
    return hm, lo, hi


def _to_display(arr, n_ch):
    """(C,H,W) normalisé -> image affichable dans [0,1]."""
    a = np.asarray(arr)
    if n_ch == 1:
        return np.clip((a[0] + 1) / 2, 0, 1)
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std  = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    return np.clip((a * std + mean).transpose(1, 2, 0), 0, 1)


def _scale_counts(scales_idx, conf):
    """Texte '128:7  64:2  32:1' pour le titre."""
    sizes, _, _ = _scale_geometry(conf)
    sc = np.asarray(scales_idx, dtype=int)
    return "  ".join(f"{sizes[k]}px:{int((sc == k).sum())}"
                     for k in range(len(sizes)))


def show_saliency(self, img, result, figsize=(18, 6)):
    conf    = self.conf
    patches = result['patches']
    scales  = np.asarray(result['scales'], dtype=int)
    scores  = np.asarray(result['scores'], dtype=float)
    gidx    = np.asarray(result['grid_idx'], dtype=int)
    n_ch    = patches.shape[1]
    M       = len(patches)
    sizes, _, _ = _scale_geometry(conf)

    img_np = _to_display(img.squeeze(0).cpu().numpy() if img.dim() == 4
                         else img.cpu().numpy(), n_ch)
    hm, lo, hi = _build_heatmap(scores, gidx, scales, conf)

    task  = list(result['label'].keys())[0]
    lbl   = result['label'][task]
    prob  = result['probs'][task]

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    fig.suptitle(f"[{self.run_name}] pred={lbl}  p={prob[lbl]:.3f}   "
                 f"{_scale_counts(scales, conf)}", fontsize=12)

    for ax, title in zip(axes, ["Image originale", "Patchs retenus", "Patchs isolés"]):
        ax.set_title(title); ax.axis('off')

    kw = dict(cmap='gray', vmin=0, vmax=1) if n_ch == 1 else {}
    axes[0].imshow(img_np, **kw)

    axes[1].imshow(img_np, **kw)
    cmap = plt.cm.hot.copy(); cmap.set_bad(alpha=0.0)
    axes[1].imshow(hm, cmap=cmap, alpha=0.5, vmin=0, vmax=1,
                   interpolation='nearest',
                   extent=[0, conf.img_size, conf.img_size, 0])
    for idx, si in zip(gidx, scales):
        r, c, p = _decode(conf, idx, si)
        axes[1].add_patch(Rectangle(
            (c, r), p, p, fill=False, lw=1.6,
            edgecolor=_SCALE_COLORS[int(si) % len(_SCALE_COLORS)]))

    # panneau 3 : les M patchs, triés par score
    order = np.argsort(scores)[::-1]
    ncol  = 5
    nrow  = int(np.ceil(M / ncol))
    P     = conf.patch_out
    grid  = (np.zeros((nrow * P, ncol * P)) if n_ch == 1
             else np.zeros((nrow * P, ncol * P, 3)))
    for k in range(M):
        rr, cc = divmod(k, ncol)
        grid[rr*P:(rr+1)*P, cc*P:(cc+1)*P] = _to_display(patches[order[k]], n_ch)
    axes[2].imshow(grid, **(kw if n_ch == 1 else {}),
                   aspect='auto', interpolation='nearest')
    for k in range(M):
        rr, cc = divmod(k, ncol)
        si = int(scales[order[k]])
        axes[2].text(cc*P + 2, rr*P + 12,
                     f"{scores[order[k]]:.3f} [{sizes[si]}px]",
                     color=_SCALE_COLORS[si % len(_SCALE_COLORS)],
                     fontsize=7, fontweight='bold',
                     bbox=dict(facecolor='black', alpha=0.55, pad=1,
                               boxstyle='round,pad=0.1'))
    plt.tight_layout()
    plt.close('all') if not plt.isinteractive() else plt.show()
    return fig


def save_saliency(self, dataset, save_dir, n_images=5, seed=42):
    """Exporte figures PNG, patchs NPY et un CSV récapitulatif."""
    os.makedirs(save_dir, exist_ok=True)
    rng  = np.random.default_rng(seed)
    idxs = rng.permutation(len(dataset))[:n_images]
    task = list(self.conf.tasks.values())[0]['name']
    sizes, _, _ = _scale_geometry(self.conf)

    csv_path = os.path.join(save_dir, f"{self.run_name}_saliency.csv")
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['img_index', 'true', 'pred', 'prob',
                    'patch_rank', 'scale_px', 'row', 'col', 'score'])

        for n, i in enumerate(idxs):
            sample = dataset[int(i)]
            img    = sample['input']
            res    = self.predict(img, return_patches=True)
            scales = np.asarray(res['scales'], dtype=int)
            scores = np.asarray(res['scores'], dtype=float)
            gidx   = np.asarray(res['grid_idx'], dtype=int)

            fig = self.show_saliency(img, res)
            fig.savefig(os.path.join(
                save_dir,
                f"{self.run_name}_saliency_{n:02d}_img{int(i)}.png"),
                dpi=110, bbox_inches='tight')
            plt.close(fig)

            np.save(os.path.join(
                save_dir, f"{self.run_name}_patches_{n:02d}_img{int(i)}.npy"),
                res['patches'])

            for rank, k in enumerate(np.argsort(scores)[::-1]):
                r, c, p = _decode(self.conf, gidx[k], scales[k])
                w.writerow([int(i), sample[task], res['label'][task],
                            f"{res['probs'][task][res['label'][task]]:.4f}",
                            rank, p, r, c, f"{scores[k]:.6f}"])

    print(f"Saillance : {n_images} images -> {save_dir}")
    print(f"CSV       : {csv_path}")
    return save_dir, csv_path


UniversalEngineMSIPS.show_saliency = show_saliency
UniversalEngineMSIPS.save_saliency = save_saliency
print("Saillance N échelles attachée à UniversalEngineMSIPS.")


Saillance N échelles attachée à UniversalEngineMSIPS.


# IX-fit + log_experiment

In [47]:
# ============================================================
# PATCH CodeCarbon — UniversalEngineMSIPS
# pip install codecarbon
# ============================================================
try:
    from codecarbon import EmissionsTracker
    _CODECARBON_AVAILABLE = True
except ImportError:
    _CODECARBON_AVAILABLE = False
    print("⚠  codecarbon non installé — pip install codecarbon")


def _fit_with_emissions_msips(self, train_loader, val_loader):
    conf      = self.conf
    log_train = Logger(conf.tasks)

    if "emissions" not in self.history:
        self.history["emissions"] = []

    import tempfile
    _co2_tmpdir = tempfile.mkdtemp()

    for epoch in range(self.start_epoch, conf.n_epoch):

        epoch_co2 = None
        if _CODECARBON_AVAILABLE:
            tracker = EmissionsTracker(
                log_level    = "error",
                output_dir   = _co2_tmpdir,
                save_to_file = False,
            )
            tracker.start()

        self._train_one_epoch(train_loader, log_train, epoch)
        log_train.compute_metric()
        log_train.print_stats(
            epoch, train=True,
            lr=f"{self.optimizer.param_groups[0]['lr']:.6f}")

        log_val = self.evaluate(val_loader, epoch=epoch)
        log_val.compute_metric()
        log_val.print_stats(0, train=False, epoch_real=epoch)

        if _CODECARBON_AVAILABLE:
            epoch_co2 = tracker.stop()
            print(f"  CO₂ époque {epoch+1} : {epoch_co2:.6f} kg")

        self.history["emissions"].append(
            round(float(epoch_co2), 8) if epoch_co2 is not None else None)

        lr_cur = self.optimizer.param_groups[0]['lr']
        self.history["lr"].append(lr_cur)

        for task in conf.tasks.values():
            t         = task['name']
            requested = task['metric']
            if isinstance(requested, str):
                requested = [requested]

            self.history["train"]["loss"].append(
                log_train.losses_epoch[t][epoch - self.start_epoch])
            for m in requested:
                self.history["train"][m].append(
                    log_train.metrics[t][epoch - self.start_epoch][m])

            self.history["val"]["loss"].append(
                log_val.losses_epoch[t][0])
            for m in requested:
                self.history["val"][m].append(
                    log_val.metrics[t][0][m])

    return self.history


def _log_experiment_with_co2_msips(self, dir_path="/kaggle/working"):
    import datetime, csv, os

    if not self.history["val"]["loss"]:
        print("Aucun résultat — lancez fit() d'abord.")
        return

    os.makedirs(dir_path, exist_ok=True)
    path = os.path.join(dir_path, f"{self.run_name}_experiment.csv")
    conf = self.conf

    summary = {}
    summary["run_name"] = self.run_name
    summary["date"]     = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    summary["n_epochs"] = len(self.history["train"]["loss"])

    for hp in ["lr", "wd", "B", "M", "I", "D", "H",
               "n_epoch_warmup", "enc_type", "pretrained",
               "scales", "patch_out"]:
        val = getattr(conf, hp, "")
        summary[hp] = "-".join(str(x) for x in val) if isinstance(val, (list, tuple)) else val

    val_losses = self.history["val"]["loss"]
    best_loss  = min(val_losses)
    summary["best_val_loss"] = round(best_loss, 6)
    summary["best_ep_loss"]  = val_losses.index(best_loss) + 1

    for m in self._metric_names:
        vals = self.history["val"].get(m, [])
        if not vals: continue
        best = max(vals)
        ep   = vals.index(best) + 1
        summary[f"best_val_{m}"] = round(best, 6)
        summary[f"best_ep_{m}"]  = ep

    summary["final_train_loss"] = round(self.history["train"]["loss"][-1], 6)
    summary["final_val_loss"]   = round(val_losses[-1], 6)
    for m in self._metric_names:
        t_vals = self.history["train"].get(m, [])
        v_vals = self.history["val"].get(m, [])
        if t_vals: summary[f"final_train_{m}"] = round(t_vals[-1], 6)
        if v_vals: summary[f"final_val_{m}"]   = round(v_vals[-1], 6)

    eff0 = self.history["efficiency"].get(0, {})
    summary["train_vram_mb"] = eff0.get("train", {}).get("vram_mb", "")
    summary["train_time_s"]  = eff0.get("train", {}).get("time_s",  "")
    summary["val_vram_mb"]   = eff0.get("val",   {}).get("vram_mb", "")
    summary["val_time_s"]    = eff0.get("val",   {}).get("time_s",  "")

    emissions = [e for e in self.history.get("emissions", [])
                 if e is not None]
    if emissions:
        summary["total_co2_kg"]     = round(sum(emissions), 8)
        summary["co2_per_epoch_kg"] = round(sum(emissions) / len(emissions), 8)
    else:
        summary["total_co2_kg"]     = ""
        summary["co2_per_epoch_kg"] = ""

    with open(path, "w", newline="") as f:
        writer = csv.writer(f)

        writer.writerow(["### RESUME"])
        writer.writerow(list(summary.keys()))
        writer.writerow(list(summary.values()))
        writer.writerow([])

        writer.writerow(["### COURBES PAR EPOQUE"])
        epoch_cols = ["epoch", "lr", "co2_kg"]
        for phase in ["train", "val"]:
            epoch_cols.append(f"{phase}_loss")
            for m in self._metric_names:
                if self.history[phase].get(m):
                    epoch_cols.append(f"{phase}_{m}")

        writer.writerow(epoch_cols)
        n_ep           = len(self.history["train"]["loss"])
        emissions_list = self.history.get("emissions", [])

        for ep in range(n_ep):
            row = [ep + 1]
            row.append(round(self.history["lr"][ep], 8)
                       if ep < len(self.history["lr"]) else "")
            co2_val = emissions_list[ep] if ep < len(emissions_list) else ""
            row.append(round(co2_val, 8)
                       if isinstance(co2_val, float) else "")
            for phase in ["train", "val"]:
                loss_vals = self.history[phase]["loss"]
                row.append(round(loss_vals[ep], 6)
                           if ep < len(loss_vals) else "")
                for m in self._metric_names:
                    m_vals = self.history[phase].get(m, [])
                    row.append(round(m_vals[ep], 6)
                               if ep < len(m_vals) else "")
            writer.writerow(row)

    print(f"Expérience sauvegardée : {path}")
    return path


# ── Application des patches sur UniversalEngineMSIPS ────────
UniversalEngineMSIPS.fit            = _fit_with_emissions_msips
UniversalEngineMSIPS.log_experiment = _log_experiment_with_co2_msips
print("Patch CodeCarbon appliqué à UniversalEngineMSIPS ✓")
print("  fit()            → tracking CO₂ par époque")
print("  log_experiment() → colonnes total_co2_kg / co2_per_epoch_kg / co2_kg")

⚠  codecarbon non installé — pip install codecarbon
Patch CodeCarbon appliqué à UniversalEngineMSIPS ✓
  fit()            → tracking CO₂ par époque
  log_experiment() → colonnes total_co2_kg / co2_per_epoch_kg / co2_kg


# X-Matrice de confusion

In [48]:
# ============================================================
# PATCH — plot_confusion_matrix pour UniversalEngineMSIPS
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import torch
import torch.nn as nn


@torch.no_grad()
def plot_confusion_matrix_msips(self, loader,
                                 class_names=None,
                                 normalize=True,
                                 save_path=None,
                                 figsize=(7, 6)):
    conf   = self.conf
    device = self.device
    net    = self.net
    net.eval()

    _net = (net.module if isinstance(net, nn.DataParallel) else net)

    all_preds  = []
    all_labels = []

    n_prep, n_prep_batch = 0, 0
    start_new_batch      = True

    for data in loader:
        imgs = data['input'].to(device)

        if start_new_batch:
            mem_patches, mem_scales, mem_pos, labels = self._init_batch()
            start_new_batch = False

        patches_iter, scales_iter, _, pos_iter = _net.ips(imgs)

        (mem_patches, mem_scales, mem_pos, labels,
         n_prep, n_prep_batch) = self._fill_batch(
            mem_patches, mem_scales, mem_pos, labels, data,
            n_prep, n_prep_batch, patches_iter, scales_iter, pos_iter)

        batch_full    = (n_prep == conf.B)
        is_last_batch = (n_prep_batch == len(loader))

        if batch_full or is_last_batch:
            if not batch_full:
                mem_patches, mem_scales, mem_pos, labels = \
                    self._shrink_batch(
                        mem_patches, mem_scales, mem_pos, labels, n_prep)

            preds      = _net(mem_patches, mem_scales, mem_pos)
            task       = list(conf.tasks.values())[0]
            t_name     = task['name']
            pred_probs = preds[t_name].detach().cpu().numpy()

            if pred_probs.ndim == 2:
                pred_labels = np.argmax(pred_probs, axis=-1)
            else:
                pred_labels = (pred_probs >= 0.5).astype(int)

            true_labels = labels[t_name].detach().cpu().numpy()
            all_preds.extend(pred_labels.tolist())
            all_labels.extend(true_labels.tolist())
            n_prep = 0; start_new_batch = True

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    n_classes  = conf.n_class

    if class_names is None:
        class_names = [f"C{i}" for i in range(n_classes)]

    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(n_classes)))

    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        cm_display = cm.astype(float) / row_sums
        title_sfx  = "(normalized, recall per class)"
    else:
        cm_display = cm
        title_sfx  = "(raw counts)"

    fig, ax = plt.subplots(figsize=figsize)
    cmap    = plt.cm.Blues
    im      = ax.imshow(cm_display, interpolation='nearest',
                        cmap=cmap,
                        vmin=0, vmax=(1.0 if normalize else cm.max()))

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    tick_marks = np.arange(n_classes)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=10)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(class_names, fontsize=10)

    thresh = cm_display.max() / 2.0
    for i in range(n_classes):
        for j in range(n_classes):
            val   = cm_display[i, j]
            text  = (f"{val:.2%}" if normalize else f"{val:d}")
            color = "white" if val > thresh else "black"
            ax.text(j, i, text,
                    ha="center", va="center",
                    color=color, fontsize=10, fontweight='bold')

    ax.set_ylabel("True label", fontsize=11)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_title(
        f"Confusion Matrix - {self.run_name}\n{title_sfx}",
        fontsize=12, pad=12)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()

    print("\nClassification Report")
    print(classification_report(
        all_labels, all_preds,
        target_names=class_names,
        zero_division=0))

    return cm


UniversalEngineMSIPS.plot_confusion_matrix = plot_confusion_matrix_msips
print("plot_confusion_matrix attaché à UniversalEngineMSIPS ✓")

plot_confusion_matrix attaché à UniversalEngineMSIPS ✓


# XI-Logger patché (kappa, f1)

In [49]:
# ============================================================
# Cellule 6 — Logger patché + Runner DDR MS-IPS
# (avec CodeCarbon + Matrice de confusion)
# ============================================================
from sklearn.metrics import cohen_kappa_score, f1_score

# ── 1. Patch Logger.compute_metric ───────────────────────────

def _patched_compute_metric(self):
    for task in self.task_dict.values():
        t      = task['name']
        losses = self.losses_it[t]
        self.losses_epoch[t].append(np.mean(losses))

        requested = task['metric']
        if isinstance(requested, str):
            requested = [requested]

        y_true     = np.array(self.y_trues[t])
        y_pred_raw = np.array(self.y_preds[t])

        epoch_metrics = {}
        for current_metric in requested:

            if current_metric == 'accuracy':
                if y_pred_raw.ndim == 2:
                    y_pred = np.argmax(y_pred_raw, axis=-1)
                else:
                    y_pred = (y_pred_raw >= 0.5).astype(int)
                epoch_metrics[current_metric] = accuracy_score(
                    y_true, y_pred)

            elif current_metric == 'kappa':
                if y_pred_raw.ndim == 2:
                    y_pred = np.argmax(y_pred_raw, axis=-1)
                else:
                    y_pred = (y_pred_raw >= 0.5).astype(int)
                try:
                    epoch_metrics[current_metric] = cohen_kappa_score(
                        y_true, y_pred, weights='quadratic')
                except Exception:
                    epoch_metrics[current_metric] = 0.0

            elif current_metric == 'f1':
                if y_pred_raw.ndim == 2:
                    y_pred = np.argmax(y_pred_raw, axis=-1)
                else:
                    y_pred = (y_pred_raw >= 0.5).astype(int)
                epoch_metrics[current_metric] = f1_score(
                    y_true, y_pred,
                    average='macro', zero_division=0)

            elif current_metric == 'auc':
                scores = (y_pred_raw[:, 1]
                          if y_pred_raw.ndim == 2 else y_pred_raw)
                epoch_metrics[current_metric] = roc_auc_score(
                    y_true, scores)

            elif current_metric == 'precision':
                if y_pred_raw.ndim == 2:
                    y_pred = np.argmax(y_pred_raw, axis=-1)
                else:
                    y_pred = (y_pred_raw >= 0.5).astype(int)
                epoch_metrics[current_metric] = precision_score(
                    y_true, y_pred, average='macro', zero_division=0)

            elif current_metric == 'recall':
                if y_pred_raw.ndim == 2:
                    y_pred = np.argmax(y_pred_raw, axis=-1)
                else:
                    y_pred = (y_pred_raw >= 0.5).astype(int)
                epoch_metrics[current_metric] = recall_score(
                    y_true, y_pred, average='macro', zero_division=0)

            elif current_metric == 'multilabel_accuracy':
                y_pred  = np.where(y_pred_raw >= 0.5, 1., 0.)
                correct = np.all(y_pred == y_true, axis=-1).sum()
                epoch_metrics[current_metric] = correct / len(y_true)

        self.metrics[t].append(epoch_metrics)
        self.losses_it[t] = []
        self.y_preds[t]   = []
        self.y_trues[t]   = []


# ── 2. Patch Logger.update ────────────────────────────────────

def _patched_update(self, next_loss, next_y_pred, next_y_true):
    for task in self.task_dict.values():
        t      = task['name']
        t_metr = task['metric']
        if isinstance(t_metr, str):
            t_metr = [t_metr]

        self.losses_it[t].append(next_loss[t])

        raw = next_y_pred[t]
        if isinstance(raw, np.ndarray):
            self.y_preds[t].extend(raw.tolist())
        else:
            self.y_preds[t].extend(raw)

        if isinstance(next_y_true[t], np.ndarray):
            self.y_trues[t].extend(next_y_true[t].tolist())
        else:
            self.y_trues[t].extend(
                next_y_true[t] if hasattr(next_y_true[t], '__iter__')
                else [next_y_true[t]])


Logger.compute_metric = _patched_compute_metric
Logger.update         = _patched_update
print("Logger.compute_metric patché ✓  (kappa QWK + f1 macro)")
print("Logger.update         patché ✓  (stockage vecteur complet)")




Logger.compute_metric patché ✓  (kappa QWK + f1 macro)
Logger.update         patché ✓  (stockage vecteur complet)


# XII-Runners

In [50]:
# ============================================================
# Runner DDR MS-IPS à trois échelles
#
# Deux runs successifs, et il faut les deux pour conclure :
#   A  patch_out = 128  comparable au run à deux échelles
#   B  patch_out =  32  pyramide stricte
# ============================================================
import os, torch, numpy as np

DataFactory._registry["msips_ddr"] = DDRDatasetMSIPS


def run_one(conf):
    print("=" * 62)
    print(f"RUN {conf.run_name}   scales={conf.scales}   patch_out={conf.patch_out}")
    print("=" * 62)

    torch.manual_seed(conf.seed)
    np.random.seed(conf.seed)

    SAVE_PATH = f"/kaggle/working/ms-ips3/ddr/{conf.run_name}"
    os.makedirs(SAVE_PATH, exist_ok=True)

    factory = DataFactory(conf, dataset="msips_ddr")
    train_loader, val_loader = factory.loaders()

    net    = MSIPSNet(DEVICE, conf)
    engine = UniversalEngineMSIPS(net, conf, DEVICE, run_name=conf.run_name)
    engine.summary()

    engine.fit(train_loader, val_loader)
    engine.summary()

    engine.save(f"{SAVE_PATH}/{conf.run_name}.pt")
    engine.save_weights(f"{SAVE_PATH}/{conf.run_name}_weights.pt")
    engine.save_history(f"{SAVE_PATH}/{conf.run_name}_history.json")
    engine.log_experiment(SAVE_PATH)

    engine.plot_confusion_matrix(
        val_loader,
        class_names=DDRDatasetMSIPS.CLASSES,
        normalize=True,
        save_path=f"{SAVE_PATH}/{conf.run_name}_confusion_matrix.png")

    engine.save_saliency(
        dataset  = factory.test_data,
        save_dir = f"{SAVE_PATH}/saliency_ddr",
        n_images = 5,
        seed     = 42)

    em = [e for e in engine.history.get("emissions", []) if e is not None]
    if em:
        print(f"CO2 total : {sum(em):.6f} kg   par époque : {sum(em)/len(em):.6f} kg")
    else:
        print("CO2 : non mesuré")

    # répartition des échelles sur le jeu de validation
    sizes = [int(s) for s in conf.scales]
    tot   = np.zeros(len(sizes))
    n_img = 0
    net.eval()
    with torch.no_grad():
        for batch in val_loader:
            x = batch["input"].to(DEVICE)
            _, ms, _, _ = net.ips(x)
            ms = ms.cpu().numpy()
            for k in range(len(sizes)):
                tot[k] += (ms == k).sum()
            n_img += ms.shape[0]
    print()
    print("Répartition des échelles sur le jeu de validation :")
    for k, s in enumerate(sizes):
        print(f"  {s:3d} px : {tot[k]/n_img:5.2f} / {conf.M} patchs "
              f"({tot[k]/tot.sum():6.1%})   a priori "
              f"{(conf.img_size//s)**2/sum((conf.img_size//z)**2 for z in sizes):6.1%}")
    return engine


engine_up  = run_one(msips3_ddr_up_conf)
engine_pyr = run_one(msips3_ddr_pyr_conf)


RUN msips3_ddr_up_v1   scales=[128, 64, 32]   patch_out=128
[DDRDatasetMSIPS] split=train |  6260 imgs | C0=3133, C1=315, C2=2238, C3=118, C4=456
[DDRDatasetMSIPS] split=valid |  2503 imgs | C0=1253, C1=126, C2=895, C3=47, C4=182
DataFactory ✓ — dataset=msips_ddr
  Train : 6260 images
  Test  : 2503 images
  Loaders créés — batch_size=8
  Train batches : 783
  Test batches  : 313
[Encoder] ImageNet (ResNet18)
DataParallel activé : 2 GPUs
UniversalEngineMSIPS instancié ✓
Optimizer : AdamW
Critères  : ['dr_grade']
History keys (train) : ['loss', 'accuracy', 'kappa', 'f1']

Run        : msips3_ddr_up_v1
Paramètres : 14.595 M



Exception in thread Thread-15 (_pin_memory_loop):


KeyboardInterrupt: 

Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
    ~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
  File "/usr/lib/python3.13/multiprocessing/resource_sharer.py", line 57, in detach
    with _resourc